# Masked mean pooling in PyTorch

Sentence-embedding models turn per-token hidden states into one vector per sequence by averaging, but padding tokens must not count.

Write `masked_mean(hidden, mask)`:

- `hidden`: float tensor `(B, T, D)` of token states.
- `mask`: tensor `(B, T)` of 0/1 (or bool). 1 marks a real token.
- Return `(B, D)`: for each sequence, the mean of `hidden` over its real tokens.

A sequence with no real tokens returns a zero vector. Use tensor ops only, no Python loops over the batch.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def masked_mean(hidden, mask):
    """hidden: (B, T, D) float, mask: (B, T) 0/1. Return (B, D) mean over real tokens."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[1.0, 2.0], [3.0, 4.0], [100.0, 100.0]]], "dtype": "float32"}, {"__nd__": [[1, 1, 0]], "dtype": "int64"}], "e": {"__nd__": [[2.0, 3.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[[1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0]], [[1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0]]], "dtype": "float32"}, {"__nd__": [[1, 0, 0], [1, 1, 1]], "dtype": "int64"}], "e": {"__nd__": [[1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[[-1.1258398294448853, -1.152360200881958, -0.2505785822868347, -0.4338788390159607, 0.8487103581428528, 0.6920092105865479], [-0.31601276993751526, -2.1152195930480957, 0.32227492332458496, -1.2633347511291504, 0.34998318552970886, 0.30813390016555786], [0.11984150856733322, 1.237657904624939, 1.1167771816253662, -0.2472776472568512, -1.3526537418365479, -1.695931315422058], [0.5666505098342896, 0.7935084104537964, 0.5988394618034363, -1.5550950765609741, -0.34136030077934265, 1.85300612449646], [0.7501894235610962, -0.5854971408843994, -0.1733970195055008, 0.18347792327404022, 1.3893661499023438, 1.5863343477249146]], [[0.946298360824585, -0.8436768054962158, -0.6135830879211426, 0.03159274160861969, -0.4926770329475403, 0.2484147548675537], [0.4396958649158478, 0.11241118609905243, 0.6407923698425293, 0.441156268119812, -0.10230965167284012, 0.7924439907073975], [-0.2896675765514374, 0.05250748619437218, 0.5228604078292847, 2.3022053241729736, -1.4688938856124878, -1.586688756942749], [-0.6730899214744568, 0.8728312253952026, 1.055357575416565, 0.1778441220521927, -0.23033547401428223, -0.3917543888092041], [0.5432946681976318, -0.39515751600265503, -0.44621720910072327, 0.7440207004547119, 1.5209795236587524, 3.4105026721954346]], [[-1.5311843156814575, -1.2341350317001343, 1.8197252750396729, -0.5515286922454834, -0.5692480206489563, 0.9199712872505188], [1.1108161211013794, 1.2898738384246826, -1.4781743288040161, 2.567232847213745, -0.4731197953224182, 0.3355507254600525], [-1.6293259859085083, -0.54974365234375, 1.1686878204345703, 0.39450275897979736, 1.9414618015289307, 0.7914981245994568], [-0.02025182731449604, -0.43716955184936523, 1.6458669900894165, -1.3601689338684082, 0.9663032293319702, 1.6247831583023071], [-0.3656197488307953, -1.3024404048919678, 0.09940346330404282, 0.44182202219963074, 0.24692639708518982, 0.07688700407743454]]], "dtype": "float32"}, {"__nd__": [[1, 1, 1, 1, 1], [1, 1, 0, 0, 0], [0, 0, 0, 0, 0]], "dtype": "int64"}], "e": {"__nd__": [[-0.0010342240566387773, -0.3643820881843567, 0.3227831721305847, -0.6632216572761536, 0.17880912125110626, 0.5487104654312134], [0.6929970979690552, -0.3656328022480011, 0.01360464096069336, 0.23637449741363525, -0.2974933385848999, 0.5204293727874756], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[[1.7340123653411865, -0.1335686445236206, -0.8351086974143982], [0.9855432510375977, -0.005227182991802692, -0.09443804621696472], [-1.123593807220459, -0.651263415813446, 0.5693069696426392], [0.1782214194536209, 1.3549144268035889, -0.011229456402361393]]], "dtype": "float32"}, {"__nd__": [[0, 1, 0, 1]], "dtype": "int64"}], "e": {"__nd__": [[0.5818823575973511, 0.6748436093330383, -0.05283375084400177]], "dtype": "float32"}}, {"a": [{"__nd__": [[[-0.8048266768455505, 0.5656090974807739, 0.6103646159172058, 0.4668835401535034, 1.9506571292877197, -1.0630985498428345, 1.1403506994247437, -0.08988206088542938], [-0.5939906239509583, -1.2439285516738892, -0.10209263116121292, -1.033548355102539, -0.3126388490200043, 0.24578548967838287, 0.37718191742897034, 1.101234793663025]], [[0.24395832419395447, 1.1646006107330322, 0.2885758578777313, 0.38659775257110596, -0.20106391608715057, -0.11792698502540588, -0.8293668627738953, -1.4072566032409668], [-1.9003455638885498, 0.13067743182182312, -0.7042941451072693, 0.3147209584712982, 0.15739288926124573, 0.3853627145290375, 0.5736545920372009, 0.997931182384491]], [[0.3016054630279541, -0.10731688141822815, 0.9984563589096069, -0.49871477484703064, 0.7611109018325806, 0.6183008551597595, -0.2993844449520111, -0.1877780705690384], [-0.1200508326292038, 0.36045974493026733, -0.3140355050563812, -1.0787080526351929, 0.24081051349639893, -1.3962273597717285, 0.11355217546224594, 1.10472571849823]], [[-1.5615617036819458, -0.3546432852745056, 1.0810725688934326, 0.13147805631160736, 1.5735375881195068, 0.7814293503761292, 0.9874365925788879, -1.4878075122833252], [1.470792531967163, 0.2756350040435791, 0.6667810678482056, -0.9943896532058716, -1.1893646717071533, -1.1959497928619385, 1.311892032623291, -0.20983858406543732]]], "dtype": "float32"}, {"__nd__": [[1, 1], [1, 1], [1, 1], [1, 1]], "dtype": "int64"}], "e": {"__nd__": [[-0.6994086503982544, -0.3391597270965576, 0.25413599610328674, -0.2833324074745178, 0.8190091252326965, -0.4086565375328064, 0.7587662935256958, 0.5056763887405396], [-0.8281936049461365, 0.6476390361785889, -0.20785914361476898, 0.3506593704223633, -0.021835513412952423, 0.1337178647518158, -0.12785613536834717, -0.20466271042823792], [0.09077731519937515, 0.1265714317560196, 0.34221041202545166, -0.788711428642273, 0.5009607076644897, -0.3889632523059845, -0.09291613101959229, 0.4584738314151764], [-0.04538458585739136, -0.03950414061546326, 0.8739268183708191, -0.4314557909965515, 0.19208645820617676, -0.20726022124290466, 1.149664282798767, -0.848823070526123]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(masked_mean, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")